In [2]:
import torch
from importlib.metadata import version
print(f'PyTorch version: {torch.__version__}')
print(f'GPU disponible : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU name       : {torch.cuda.get_device_name(0)}')
    print(f'GPU memory     : {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')

PyTorch version: 2.6.0+cu124
GPU disponible : True
GPU name       : NVIDIA GeForce GTX 1650
GPU memory     : 4.0 GB


## Étape 1 — Chargement du KG dans PyKEEN


In [3]:
import os
import pandas as pd
from pykeen.triples import TriplesFactory

KG_DIR = r'D:\Memoire\LLMKGER\kg_final'

# Charger kg_train.tsv et EXCLURE les follows
df = pd.read_csv(os.path.join(KG_DIR, 'kg_train.tsv'),
                 sep='\t', header=0, names=['head','relation','tail'])

# Garder seulement les triplets KG (hors follows)
df_kg = df[df['relation'] != 'follows'].copy()

print(f'Triplets totaux    : {len(df):,}')
print(f'Triplets KG only   : {len(df_kg):,}')
print(f'Relations distinctes : {df_kg["relation"].nunique()}')
print()
print(df_kg['relation'].value_counts())

Triplets totaux    : 5,693,239
Triplets KG only   : 648,349
Relations distinctes : 6

relation
has_concept        251334
belongs_to         211691
watches            112038
prerequisite_of     59751
taught_by           10742
offered_by           2793
Name: count, dtype: int64


In [4]:
tf = TriplesFactory.from_labeled_triples(
    triples=df_kg[['head','relation','tail']].values,
    create_inverse_triples=False
)

train_tf, valid_tf, test_tf = tf.split([0.85, 0.075, 0.075], random_state=42)

print(f'Train triplets : {train_tf.num_triples:,}')
print(f'Valid triplets : {valid_tf.num_triples:,}')
print(f'Test  triplets : {test_tf.num_triples:,}')
print(f'Entites        : {tf.num_entities:,}')
print(f'Relations      : {tf.num_relations:,}')

Train triplets : 551,096
Valid triplets : 48,626
Test  triplets : 48,627
Entites        : 141,591
Relations      : 6


## Étape 2 — Entraînement des 5 modèles KGE


In [5]:
import torch
from pykeen.pipeline import pipeline
import json, os

RESULTS_DIR = r'D:\Memoire\LLMKGER\kge_results'
os.makedirs(RESULTS_DIR, exist_ok=True)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# Configuration commune
COMMON_CONFIG = {
    'num_epochs'      : 100,
    'embedding_dim'   : 128,
    'batch_size'      : 1024,
    'device'          : DEVICE,
    'random_seed'     : 42,
}

# Configurations specifiques par modele
MODELS_CONFIG = {
    'TransE': {
        'model': 'TransE',
        'model_kwargs': {'embedding_dim': COMMON_CONFIG['embedding_dim'],
                         'scoring_fct_norm': 1},
        'loss': 'MarginRankingLoss',
        'loss_kwargs': {'margin': 1.0},
        'optimizer': 'Adam',
        'optimizer_kwargs': {'lr': 0.001},
    },
    'RotatE': {
        'model': 'RotatE',
        'model_kwargs': {'embedding_dim': COMMON_CONFIG['embedding_dim']},
        'loss': 'NSSALoss',
        'optimizer': 'Adam',
        'optimizer_kwargs': {'lr': 0.001},
    },
    'ComplEx': {
        'model': 'ComplEx',
        'model_kwargs': {'embedding_dim': COMMON_CONFIG['embedding_dim']},
        'loss': 'SoftplusLoss',
        'optimizer': 'Adagrad',
        'optimizer_kwargs': {'lr': 0.1},
    },
    'DistMult': {
        'model': 'DistMult',
        'model_kwargs': {'embedding_dim': COMMON_CONFIG['embedding_dim']},
        'loss': 'SoftplusLoss',
        'optimizer': 'Adagrad',
        'optimizer_kwargs': {'lr': 0.1},
    },
    'TransR': {
        'model': 'TransR',
        'model_kwargs': {'embedding_dim': COMMON_CONFIG['embedding_dim'],
                         'relation_dim': 64},
        'loss': 'MarginRankingLoss',
        'loss_kwargs': {'margin': 1.0},
        'optimizer': 'Adam',
        'optimizer_kwargs': {'lr': 0.001},
    },
}

print(f'Device : {DEVICE}')
print(f'Modeles a entrainer : {list(MODELS_CONFIG.keys())}')
print(f'Epochs : {COMMON_CONFIG["num_epochs"]}')
print(f'Embedding dim : {COMMON_CONFIG["embedding_dim"]}')


Device : cuda
Modeles a entrainer : ['TransE', 'RotatE', 'ComplEx', 'DistMult', 'TransR']
Epochs : 100
Embedding dim : 128


In [ ]:
all_results = {}

for model_name, cfg in MODELS_CONFIG.items():
    print(f'\n{"="*50}')
    print(f'  Entrainement : {model_name}')
    print(f'{"="*50}')

    try:
        result = pipeline(
            training   = train_tf,
            validation = valid_tf,
            testing    = test_tf,
            model      = cfg['model'],
            model_kwargs        = cfg.get('model_kwargs', {}),
            loss                = cfg.get('loss', 'MarginRankingLoss'),
            loss_kwargs         = cfg.get('loss_kwargs', {}),
            optimizer           = cfg.get('optimizer', 'Adam'),
            optimizer_kwargs    = cfg.get('optimizer_kwargs', {'lr': 0.001}),
            training_kwargs = {
                'num_epochs' : COMMON_CONFIG['num_epochs'],
                'batch_size' : COMMON_CONFIG['batch_size'],
            },
            evaluator_kwargs = {'filtered': True},
            random_seed = COMMON_CONFIG['random_seed'],
            device      = COMMON_CONFIG['device'],
        )

        # Extraire les metriques
        metrics = result.metric_results.to_dict()
        mrr     = metrics.get('both.realistic.inverse_harmonic_mean_rank', 0)
        h1      = metrics.get('both.realistic.hits_at_1', 0)
        h3      = metrics.get('both.realistic.hits_at_3', 0)
        h10     = metrics.get('both.realistic.hits_at_10', 0)

        all_results[model_name] = {
            'MRR'    : round(mrr, 4),
            'Hit@1'  : round(h1,  4),
            'Hit@3'  : round(h3,  4),
            'Hit@10' : round(h10, 4),
        }

        print(f'  MRR    : {mrr:.4f}')
        print(f'  Hit@1  : {h1:.4f}')
        print(f'  Hit@3  : {h3:.4f}')
        print(f'  Hit@10 : {h10:.4f}')

        # Sauvegarder le modele
        model_dir = os.path.join(RESULTS_DIR, model_name)
        result.save_to_directory(model_dir)
        print(f'  Sauvegarde -> {model_dir}')

    except Exception as e:
        print(f'  ERREUR : {e}')
        all_results[model_name] = {'erreur': str(e)}

print('\n=== Entrainement termine ===')



  Entrainement : TransE


Training epochs on cuda:0:   0%|          | 0/100 [00:00<?, ?epoch/s]

Training batches on cuda:0:   0%|          | 0.00/539 [00:00<?, ?batch/s]

Training batches on cuda:0:   0%|          | 0.00/539 [00:00<?, ?batch/s]

Training batches on cuda:0:   0%|          | 0.00/539 [00:00<?, ?batch/s]

Training batches on cuda:0:   0%|          | 0.00/539 [00:00<?, ?batch/s]

In [6]:
print(train_tf.num_triples)
print(list(all_results.keys()))

551096


NameError: name 'all_results' is not defined

In [ ]:
import torch, os
from pykeen.pipeline import pipeline

RESULTS_DIR = r'D:\Memoire\LLMKGER\kge_results'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

all_results = {}  # on recrée le dict

print('Entraînement TransR...')
result_transr = pipeline(
    training         = train_tf,
    validation       = valid_tf,
    testing          = test_tf,
    model            = 'TransR',
    model_kwargs     = {'embedding_dim': 128, 'relation_dim': 64},
    loss             = 'MarginRankingLoss',
    loss_kwargs      = {'margin': 1.0},
    optimizer        = 'Adam',
    optimizer_kwargs = {'lr': 0.001},
    training_kwargs  = {'num_epochs': 100, 'batch_size': 1024},
    evaluator_kwargs = {'filtered': True, 'batch_size': 16},
    random_seed      = 42,
    device           = DEVICE,
)

metrics = result_transr.metric_results.to_dict()
mrr = metrics.get('both.realistic.inverse_harmonic_mean_rank', 0)
h1  = metrics.get('both.realistic.hits_at_1', 0)
h3  = metrics.get('both.realistic.hits_at_3', 0)
h10 = metrics.get('both.realistic.hits_at_10', 0)

all_results['TransR'] = {
    'MRR'   : round(mrr, 4),
    'Hit@1' : round(h1,  4),
    'Hit@3' : round(h3,  4),
    'Hit@10': round(h10, 4),
}

print(f'TransR — MRR: {mrr:.4f} | Hit@1: {h1:.4f} | Hit@3: {h3:.4f} | Hit@10: {h10:.4f}')
result_transr.save_to_directory(os.path.join(RESULTS_DIR, 'TransR'))
print('TransR sauvegardé ✅')

INFO:pykeen.pipeline.api:Using device: cuda
INFO:pykeen.nn.representation:Inferred unique=False for Embedding()
INFO:pykeen.nn.representation:Inferred unique=False for Embedding()
INFO:pykeen.nn.representation:Inferred unique=False for Embedding()


Entraînement TransR...


Training epochs on cuda:0:   0%|          | 0/100 [00:00<?, ?epoch/s]

Training batches on cuda:0:   0%|          | 0.00/539 [00:00<?, ?batch/s]

Training batches on cuda:0:   0%|          | 0.00/539 [00:00<?, ?batch/s]

Training batches on cuda:0:   0%|          | 0.00/539 [00:00<?, ?batch/s]

... (98 outputs left out)

INFO:pykeen.evaluation.evaluator:Evaluation took 749.77s seconds


TransR — MRR: 0.0000 | Hit@1: 0.0000 | Hit@3: 0.0000 | Hit@10: 0.0000


INFO:pykeen.triples.triples_factory:Stored TriplesFactory(num_entities=141591, num_relations=6, create_inverse_triples=False, num_triples=551096) to file:///D:/Memoire/LLMKGER/kge_results/TransR/training_triples
INFO:pykeen.pipeline.api:Saved to directory: D:\Memoire\LLMKGER\kge_results\TransR


TransR sauvegardé ✅


In [11]:
all_results = {}  # ← ajoute cette ligne en premier

import os, json
import pandas as pd

RESULTS_DIR = r'D:\Memoire\LLMKGER\kge_results'

for model_name in ['TransE', 'RotatE', 'ComplEx', 'DistMult']:
    metrics_path = os.path.join(RESULTS_DIR, model_name, 'results.json')
    if os.path.exists(metrics_path):
        with open(metrics_path, 'r') as f:
            data = json.load(f)
        metrics = data.get('metrics', {})
        mrr = metrics.get('both.realistic.inverse_harmonic_mean_rank', 0)
        h1  = metrics.get('both.realistic.hits_at_1', 0)
        h3  = metrics.get('both.realistic.hits_at_3', 0)
        h10 = metrics.get('both.realistic.hits_at_10', 0)
        all_results[model_name] = {
            'MRR'   : round(mrr, 4),
            'Hit@1' : round(h1,  4),
            'Hit@3' : round(h3,  4),
            'Hit@10': round(h10, 4),
        }
        print(f'{model_name} rechargé ✅')
    else:
        print(f'{model_name} — fichier introuvable : {metrics_path}')

print(f'\nModèles chargés : {list(all_results.keys())}')

TransE rechargé ✅
RotatE rechargé ✅
ComplEx rechargé ✅
DistMult rechargé ✅

Modèles chargés : ['TransE', 'RotatE', 'ComplEx', 'DistMult']


## Étape 3 — Comparaison des modèles


In [15]:
import os, json
import pandas as pd

RESULTS_DIR = r'D:\Memoire\LLMKGER\kge_results'
all_results = {}

for model_name in ['TransE', 'RotatE', 'ComplEx', 'DistMult']:
    metrics_path = os.path.join(RESULTS_DIR, model_name, 'results.json')
    if os.path.exists(metrics_path):
        with open(metrics_path, 'r') as f:
            data = json.load(f)
        
        # Navigation correcte dans la structure imbriquée
        m = data['metrics']['both']['realistic']
        
        all_results[model_name] = {
            'MRR'   : round(m['inverse_harmonic_mean_rank'], 4),
            'Hit@1' : round(m['hits_at_1'], 4),
            'Hit@3' : round(m['hits_at_3'], 4),
            'Hit@10': round(m['hits_at_10'], 4),
        }
        print(f'{model_name} rechargé ✅')
    else:
        print(f'{model_name} — fichier introuvable')

# Tableau comparatif
df_results = pd.DataFrame(all_results).T
df_results = df_results.sort_values('MRR', ascending=False)

print('\n' + '=' * 55)
print('  Comparaison des modèles KGE')
print('=' * 55)
print(df_results.to_string())
print('=' * 55)

best_model = df_results.index[0]
print(f'\n🏆 Meilleur modèle : {best_model}')
print(f'   MRR    : {df_results.loc[best_model, "MRR"]}')
print(f'   Hit@1  : {df_results.loc[best_model, "Hit@1"]}')
print(f'   Hit@3  : {df_results.loc[best_model, "Hit@3"]}')
print(f'   Hit@10 : {df_results.loc[best_model, "Hit@10"]}')

df_results.to_csv(os.path.join(RESULTS_DIR, 'kge_comparison.csv'))
print(f'\nTableau sauvegardé ✅')

TransE rechargé ✅
RotatE rechargé ✅
ComplEx rechargé ✅
DistMult rechargé ✅

  Comparaison des modèles KGE
             MRR   Hit@1   Hit@3  Hit@10
RotatE    0.1975  0.1350  0.2097  0.3167
TransE    0.0734  0.0147  0.0920  0.1768
ComplEx   0.0205  0.0102  0.0204  0.0396
DistMult  0.0007  0.0004  0.0006  0.0009

🏆 Meilleur modèle : RotatE
   MRR    : 0.1975
   Hit@1  : 0.135
   Hit@3  : 0.2097
   Hit@10 : 0.3167

Tableau sauvegardé ✅


In [13]:
import json, os

RESULTS_DIR = r'D:\Memoire\LLMKGER\kge_results'

# Inspecter la structure du results.json de TransE
with open(os.path.join(RESULTS_DIR, 'TransE', 'results.json'), 'r') as f:
    data = json.load(f)

# Afficher les clés de premier niveau
print("Clés niveau 1 :", list(data.keys()))

# Afficher la structure complète des métriques
import pprint
if 'metrics' in data:
    print("\nStructure metrics :")
    pprint.pprint(data['metrics'])
elif 'results' in data:
    print("\nStructure results :")
    pprint.pprint(data['results'])
else:
    print("\nStructure complète :")
    pprint.pprint(data)

Clés niveau 1 : ['losses', 'metrics', 'times']

Structure metrics :
{'both': {'optimistic': {'adjusted_arithmetic_mean_rank': 0.03983431803515784,
                         'adjusted_arithmetic_mean_rank_index': 0.9601792598453313,
                         'adjusted_geometric_mean_rank_index': 0.996782403264975,
                         'adjusted_hits_at_k': 0.17670572594675465,
                         'adjusted_inverse_harmonic_mean_rank': 0.07332290857675253,
                         'arithmetic_mean_rank': 2816.9408353383924,
                         'count': 97254.0,
                         'geometric_mean_rank': 168.41631498360996,
                         'harmonic_mean_rank': 13.623162238565607,
                         'hits_at_1': 0.014703765397824253,
                         'hits_at_10': 0.17676393773006766,
                         'hits_at_3': 0.09204762786106484,
                         'hits_at_5': 0.12623645299936248,
                         'inverse_arithmetic_mean

## Étape 4 — Extraction de S_KG (scores pour la Hybrid Function)


In [6]:
import torch
import numpy as np
import pandas as pd
import os

RESULTS_DIR = r'D:\Memoire\LLMKGER\kge_results'
KG_DIR      = r'D:\Memoire\LLMKGER\kg_final'
best_model  = 'RotatE'

# Recharger le modele sauvegarde
trained_model = torch.load(
    os.path.join(RESULTS_DIR, best_model, 'trained_model.pkl'),
    map_location='cpu',
    weights_only=False   # ← correction PyTorch 2.6
)
trained_model.eval()
print(f'Modele {best_model} charge ✅')

# Charger les interactions train pour extraire S_KG(user, course)
interactions_train = pd.read_csv(
    os.path.join(KG_DIR, 'interactions_train.tsv'),
    sep='\t', header=0, names=['user_id','relation','course_id']
)
print(f'Interactions train : {len(interactions_train):,}')
print(interactions_train.head(3))

Modele RotatE charge ✅
Interactions train : 5,044,890
      user_id relation course_id
0  U_12868687  follows  C_934517
1  U_12868687  follows  C_694168
2  U_12868687  follows  C_814402


In [8]:
import os

# Voir ce qu'il y a dans le dossier training_triples
training_triples_path = os.path.join(RESULTS_DIR, 'RotatE', 'training_triples')
print(f'Est un dossier : {os.path.isdir(training_triples_path)}')
print(f'Contenu :')
for f in os.listdir(training_triples_path):
    print(f'  {f}')

Est un dossier : True
Contenu :
  base.pth
  entity_to_id.tsv.gz
  numeric_triples.tsv.gz
  relation_to_id.tsv.gz


In [9]:
import pandas as pd
import os

training_triples_path = os.path.join(RESULTS_DIR, 'RotatE', 'training_triples')

# Charger entity2id et relation2id directement
entity2id_df   = pd.read_csv(
    os.path.join(training_triples_path, 'entity_to_id.tsv.gz'),
    sep='\t', compression='gzip'
)
relation2id_df = pd.read_csv(
    os.path.join(training_triples_path, 'relation_to_id.tsv.gz'),
    sep='\t', compression='gzip'
)

print(f'Entités   : {len(entity2id_df):,}')
print(f'Relations : {len(relation2id_df):,}')
print(f'\nColonnes entity2id   : {entity2id_df.columns.tolist()}')
print(f'Colonnes relation2id : {relation2id_df.columns.tolist()}')
print(f'\nExemples entités :')
print(entity2id_df.head(3))
print(f'\nRelations :')
print(relation2id_df)

Entités   : 141,591
Relations : 6

Colonnes entity2id   : ['id', 'label']
Colonnes relation2id : ['id', 'label']

Exemples entités :
   id                             label
0   0  00059EBD1371A6959C33DC5901307461
1   1  0005D1DC01B4EF8F9C33DC5901307461
2   2  000BD50F6730291F9C33DC5901307461

Relations :
   id            label
0   0       belongs_to
1   1      has_concept
2   2       offered_by
3   3  prerequisite_of
4   4        taught_by
5   5          watches


In [10]:
import torch
import numpy as np
import pandas as pd
import os

# Construire les mappings
entity2id   = dict(zip(entity2id_df['label'], entity2id_df['id']))
relation2id = dict(zip(relation2id_df['label'], relation2id_df['id']))

print(f'entity2id size   : {len(entity2id):,}')
print(f'relation2id      : {relation2id}')

# Vérifier quelques entités
sample_user   = interactions_train['user_id'].iloc[0]
sample_course = interactions_train['course_id'].iloc[0]

print(f'\nTest user   : {sample_user} → in KG : {sample_user in entity2id}')
print(f'Test course : {sample_course} → in KG : {sample_course in entity2id}')

# Extraire les embeddings de toutes les entités
with torch.no_grad():
    all_embeddings = trained_model.entity_representations[0](indices=None)
    
print(f'\nShape embeddings : {all_embeddings.shape}')
# (141591, 128) attendu

entity2id size   : 141,591
relation2id      : {'belongs_to': 0, 'has_concept': 1, 'offered_by': 2, 'prerequisite_of': 3, 'taught_by': 4, 'watches': 5}

Test user   : U_12868687 → in KG : False
Test course : C_934517 → in KG : True

Shape embeddings : torch.Size([141591, 128])


In [11]:
import torch
import torch.nn.functional as F
import numpy as np
import pandas as pd
import os
from collections import defaultdict
from tqdm import tqdm

# ── 1. Charger les triplets du KG pour reconstruire les chemins ────────────
kg_path = os.path.join(KG_DIR, 'kg_train.tsv')
df_kg = pd.read_csv(kg_path, sep='\t', header=0, names=['head','relation','tail'])
df_kg = df_kg[df_kg['relation'] != 'follows']

print(f'Triplets KG (hors follows) : {len(df_kg):,}')

# ── 2. Construire mapping user → videos regardées ─────────────────────────
watches_df = df_kg[df_kg['relation'] == 'watches']
user_to_videos = defaultdict(set)
for _, row in tqdm(watches_df.iterrows(), total=len(watches_df), desc='user→videos'):
    user_to_videos[row['head']].add(row['tail'])

print(f'Users avec videos : {len(user_to_videos):,}')

# ── 3. Construire mapping video → concepts ────────────────────────────────
hasconcept_df = df_kg[df_kg['relation'] == 'has_concept']
video_to_concepts = defaultdict(set)
for _, row in hasconcept_df.iterrows():
    video_to_concepts[row['head']].add(row['tail'])

# ── 4. Construire mapping concept → courses ───────────────────────────────
belongsto_df = df_kg[df_kg['relation'] == 'belongs_to']
concept_to_courses = defaultdict(set)
for _, row in belongsto_df.iterrows():
    concept_to_courses[row['head']].add(row['tail'])

print(f'Concepts avec cours : {len(concept_to_courses):,}')

Triplets KG (hors follows) : 648,349


user→videos: 100%|██████████| 112038/112038 [00:16<00:00, 6907.04it/s]


Users avec videos : 11,223
Concepts avec cours : 100,059


In [12]:
# ── 5. Fonction pour obtenir l'embedding d'une entité ─────────────────────
def get_embedding(entity_id):
    if entity_id not in entity2id:
        return None
    idx = entity2id[entity_id]
    with torch.no_grad():
        emb = all_embeddings[idx]
    return emb

# ── 6. Calculer S_KG(user, course) via chemin sémantique ──────────────────
# S_KG = cosine similarity moyenne entre embedding du cours cible
#        et embeddings des cours liés aux videos regardées par l'user

def compute_s_kg(user_id, course_id):
    # Embedding du cours cible
    course_emb = get_embedding(course_id)
    if course_emb is None:
        return 0.0

    # Récupérer les videos regardées par l'user
    videos = user_to_videos.get(user_id, set())
    if not videos:
        return 0.0

    # Récupérer les concepts liés aux videos
    related_concepts = set()
    for vid in videos:
        related_concepts.update(video_to_concepts.get(vid, set()))

    if not related_concepts:
        return 0.0

    # Récupérer les cours liés aux concepts
    related_courses = set()
    for concept in related_concepts:
        related_courses.update(concept_to_courses.get(concept, set()))

    # Enlever le cours cible lui-même
    related_courses.discard(course_id)

    if not related_courses:
        return 0.0

    # Calculer similarité cosine moyenne
    similarities = []
    for rc in list(related_courses)[:50]:  # max 50 pour la vitesse
        rc_emb = get_embedding(rc)
        if rc_emb is not None:
            sim = F.cosine_similarity(
                course_emb.unsqueeze(0),
                rc_emb.unsqueeze(0)
            ).item()
            similarities.append(sim)

    return float(np.mean(similarities)) if similarities else 0.0

# ── 7. Test sur quelques paires ───────────────────────────────────────────
print('Test S_KG sur 5 paires :')
for i in range(5):
    uid = interactions_train['user_id'].iloc[i]
    cid = interactions_train['course_id'].iloc[i]
    score = compute_s_kg(uid, cid)
    print(f'  S_KG({uid}, {cid}) = {score:.4f}')

Test S_KG sur 5 paires :
  S_KG(U_12868687, C_934517) = 0.0000
  S_KG(U_12868687, C_694168) = 0.0000
  S_KG(U_12868687, C_814402) = 0.0000
  S_KG(U_12868687, C_696586) = 0.0000
  S_KG(U_12868687, C_947241) = 0.0000


In [13]:
# Diagnostic complet
uid = 'U_12868687'
cid = 'C_934517'

print(f'User dans user_to_videos : {uid in user_to_videos}')
print(f'Course dans entity2id    : {cid in entity2id}')

# Chercher quelques users qui SONT dans user_to_videos
sample_users_with_videos = list(user_to_videos.keys())[:5]
print(f'\nExemples users dans user_to_videos : {sample_users_with_videos}')

# Vérifier si ces users sont dans interactions_train
print(f'\nCes users sont dans interactions_train ?')
for u in sample_users_with_videos:
    in_train = u in interactions_train['user_id'].values
    print(f'  {u} → {in_train}')

# Compter combien de users de interactions_train sont dans user_to_videos
users_in_train = set(interactions_train['user_id'].unique())
users_in_kg    = set(user_to_videos.keys())
overlap = users_in_train & users_in_kg

print(f'\nUsers dans interactions_train : {len(users_in_train):,}')
print(f'Users dans user_to_videos    : {len(users_in_kg):,}')
print(f'Overlap (intersection)       : {len(overlap):,}')

User dans user_to_videos : False
Course dans entity2id    : True

Exemples users dans user_to_videos : ['U_23968068', 'U_31017024', 'U_27198938', 'U_26340059', 'U_18183768']

Ces users sont dans interactions_train ?
  U_23968068 → True
  U_31017024 → True
  U_27198938 → True
  U_26340059 → True
  U_18183768 → True

Users dans interactions_train : 106,735
Users dans user_to_videos    : 11,223
Overlap (intersection)       : 11,223


In [2]:
import torch
import torch.nn.functional as F
import numpy as np
import pandas as pd
import os
from collections import defaultdict

RESULTS_DIR = r'D:\Memoire\LLMKGER\kge_results'
KG_DIR      = r'D:\Memoire\LLMKGER\kg_final'

# ── 1. Recharger le modèle RotatE ─────────────────────────────────────────
trained_model = torch.load(
    os.path.join(RESULTS_DIR, 'RotatE', 'trained_model.pkl'),
    map_location='cpu', weights_only=False
)
trained_model.eval()
print('RotatE chargé ✅')

# ── 2. Recharger entity2id ────────────────────────────────────────────────
training_triples_path = os.path.join(RESULTS_DIR, 'RotatE', 'training_triples')
entity2id_df = pd.read_csv(
    os.path.join(training_triples_path, 'entity_to_id.tsv.gz'),
    sep='\t', compression='gzip'
)
entity2id = dict(zip(entity2id_df['label'], entity2id_df['id']))
print(f'entity2id : {len(entity2id):,} ✅')

# ── 3. Extraire tous les embeddings ───────────────────────────────────────
with torch.no_grad():
    all_embeddings = trained_model.entity_representations[0](indices=None)
print(f'Embeddings shape : {all_embeddings.shape} ✅')

# ── 4. Charger les triplets KG (hors follows) ─────────────────────────────
df_kg = pd.read_csv(
    os.path.join(KG_DIR, 'kg_train.tsv'),
    sep='\t', header=0, names=['head','relation','tail']
)
df_kg = df_kg[df_kg['relation'] != 'follows']
print(f'Triplets KG : {len(df_kg):,} ✅')

# ── 5. Construire les mappings (version rapide) ───────────────────────────
watches_df    = df_kg[df_kg['relation'] == 'watches']
hasconcept_df = df_kg[df_kg['relation'] == 'has_concept']
belongsto_df  = df_kg[df_kg['relation'] == 'belongs_to']

user_to_videos     = watches_df.groupby('head')['tail'].apply(set).to_dict()
video_to_concepts  = hasconcept_df.groupby('head')['tail'].apply(set).to_dict()
concept_to_courses = belongsto_df.groupby('head')['tail'].apply(set).to_dict()
print(f'user_to_videos     : {len(user_to_videos):,} ✅')
print(f'video_to_concepts  : {len(video_to_concepts):,} ✅')
print(f'concept_to_courses : {len(concept_to_courses):,} ✅')

# ── 6. Charger interactions train ─────────────────────────────────────────
interactions_train = pd.read_csv(
    os.path.join(KG_DIR, 'interactions_train.tsv'),
    sep='\t', header=0, names=['user_id','relation','course_id']
)
user_to_courses_train = interactions_train.groupby('user_id')['course_id'].apply(set).to_dict()
print(f'interactions_train     : {len(interactions_train):,} ✅')
print(f'user_to_courses_train  : {len(user_to_courses_train):,} ✅')

# ── 7. Fonction embedding ──────────────────────────────────────────────────
def get_embedding(entity_id):
    if entity_id not in entity2id:
        return None
    idx = entity2id[entity_id]
    with torch.no_grad():
        return all_embeddings[idx]

# ── 8. Fonction S_KG v2 ───────────────────────────────────────────────────
def compute_s_kg_v2(user_id, course_id):
    course_emb = get_embedding(course_id)
    if course_emb is None:
        return 0.0

    related_courses = set()

    # Chemin 1 : via videos → concepts → cours
    for vid in user_to_videos.get(user_id, set()):
        for concept in video_to_concepts.get(vid, set()):
            related_courses.update(concept_to_courses.get(concept, set()))

    # Chemin 2 : fallback via cours déjà suivis
    if not related_courses:
        related_courses = user_to_courses_train.get(user_id, set())

    related_courses.discard(course_id)
    if not related_courses:
        return 0.0

    similarities = []
    for rc in list(related_courses)[:50]:
        rc_emb = get_embedding(rc)
        if rc_emb is not None:
            sim = F.cosine_similarity(
                course_emb.unsqueeze(0),
                rc_emb.unsqueeze(0)
            ).item()
            similarities.append(sim)

    return float(np.mean(similarities)) if similarities else 0.0

# ── 9. Test sur 5 paires ──────────────────────────────────────────────────
print('\nTest S_KG v2 sur 5 paires :')
for i in range(5):
    uid = interactions_train['user_id'].iloc[i]
    cid = interactions_train['course_id'].iloc[i]
    score = compute_s_kg_v2(uid, cid)
    print(f'  S_KG({uid}, {cid}) = {score:.4f}')

RotatE chargé ✅
entity2id : 141,591 ✅
Embeddings shape : torch.Size([141591, 128]) ✅
Triplets KG : 648,349 ✅
user_to_videos     : 11,223 ✅
video_to_concepts  : 13,485 ✅
concept_to_courses : 100,059 ✅
interactions_train     : 5,044,890 ✅
user_to_courses_train  : 106,735 ✅

Test S_KG v2 sur 5 paires :


RuntimeError: expected common dtype to be floating point, yet common dtype is ComplexFloat

In [3]:
# Correction get_embedding pour RotatE (embeddings complexes)
def get_embedding(entity_id):
    if entity_id not in entity2id:
        return None
    idx = entity2id[entity_id]
    with torch.no_grad():
        emb = all_embeddings[idx]
        if emb.is_complex():
            emb = emb.real
        return emb

# Test sur 5 paires
print('Test S_KG v2 sur 5 paires :')
for i in range(5):
    uid = interactions_train['user_id'].iloc[i]
    cid = interactions_train['course_id'].iloc[i]
    score = compute_s_kg_v2(uid, cid)
    print(f'  S_KG({uid}, {cid}) = {score:.4f}')

Test S_KG v2 sur 5 paires :
  S_KG(U_12868687, C_934517) = -0.0063
  S_KG(U_12868687, C_694168) = -0.0114
  S_KG(U_12868687, C_814402) = 0.0626
  S_KG(U_12868687, C_696586) = 0.0566
  S_KG(U_12868687, C_947241) = 0.0334


In [4]:
from tqdm import tqdm

# Calculer S_KG pour un échantillon représentatif
# (on prend 200K paires pour ne pas bloquer trop longtemps)
SAMPLE_SIZE = 200_000
sample_interactions = interactions_train.sample(n=SAMPLE_SIZE, random_state=42)

print(f'Calcul S_KG pour {SAMPLE_SIZE:,} paires...')

s_kg_records = []
for _, row in tqdm(sample_interactions.iterrows(), total=SAMPLE_SIZE):
    uid = row['user_id']
    cid = row['course_id']
    score = compute_s_kg_v2(uid, cid)
    s_kg_records.append({
        'user_id'  : uid,
        'course_id': cid,
        'S_KG'     : score
    })

df_skg = pd.DataFrame(s_kg_records)

print(f'\n✅ S_KG calculé pour {len(df_skg):,} paires')
print(f'\nStatistiques S_KG :')
print(df_skg['S_KG'].describe())

# Sauvegarder
skg_path = os.path.join(RESULTS_DIR, 'S_KG_scores.csv')
df_skg.to_csv(skg_path, index=False)
print(f'\nS_KG sauvegardé → {skg_path} ✅')

Calcul S_KG pour 200,000 paires...


100%|██████████| 200000/200000 [19:18<00:00, 172.62it/s]



✅ S_KG calculé pour 200,000 paires

Statistiques S_KG :
count    200000.000000
mean          0.046899
std           0.078345
min          -0.329637
25%           0.010527
50%           0.033528
75%           0.065394
max           0.999386
Name: S_KG, dtype: float64

S_KG sauvegardé → D:\Memoire\LLMKGER\kge_results\S_KG_scores.csv ✅
